# 9-3절 연습 문제 풀이

이 노트북은 9-3절 연습 문제(9-9 ~ 9-11)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch09/09-03_example.ipynb`를 참고한다.
- 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

## 공통 준비 — 기본 Seq2Seq

In [1]:
# 환경 설정 (code_reference 모듈 임포트 경로와 시각화 설정)
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

common.set_korean_plot_env()
viz.configure(save_grayscale=False)

SEED = 42
common.set_seed(SEED, deterministic=True)
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# 날짜 데이터 생성 함수 (본문 예제와 동일)
import random
import string
from datetime import datetime, timedelta

src_formats = [
    '%d %B %Y',     # 01 February 2026
    '%d %b %Y',     # 01 Feb 2026
    '%B %d, %Y',    # February 01, 2026
    '%b %d, %Y',    # Feb 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]


def choice_random_dates(sample_size=1, start_date=None, end_date=None):
    if start_date is None:
        start_date = datetime(1900, 1, 1)
    if end_date is None:
        end_date = datetime(2050, 12, 31)
    days_between = (end_date - start_date).days
    datetime_list = []
    for _ in range(sample_size):
        days_after = random.randrange(days_between)
        datetime_list.append(start_date + timedelta(days=days_after))
    return datetime_list


def generate_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(date.strftime(src_format))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


def add_random_noise(text, max_length=40):
    noise_chars = (
        string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
    )
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = random.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(random.choices(noise_chars, k=prefix_length))
    if remaining > 0:
        suffix_length = random.randint(0, remaining)
        suffix = ''.join(random.choices(noise_chars, k=suffix_length))
    return prefix + text + suffix


def generate_noisy_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(add_random_noise(date.strftime(src_format)))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


common.set_seed(SEED, deterministic=True)
date_list = choice_random_dates(4000)
src_dates, tgt_dates = generate_datepairs(date_list)
print(f'{src_dates[0]!r} -> {tgt_dates[0]!r}')

'25 September 2014' -> '2014-9-25'


In [3]:
# 어휘 사전과 데이터셋 ([코드 9-1], [코드 9-12])
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SOS_TOKEN, EOS_TOKEN, PAD_TOKEN = '<sos>', '<eos>', '<pad>'
SOS_IDX, EOS_IDX, PAD_IDX = 0, 1, 2
special_tokens = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, PAD_TOKEN: PAD_IDX}


class Vocab:
    def __init__(self, sequence_list, special_tokens):
        tokens = set()
        for sequence in sequence_list:
            tokens.update(sequence)
        self.vocab = {}
        self.vocab.update(special_tokens)
        idx_start = len(special_tokens)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + idx_start
        self.itos = {idx: token for token, idx in self.vocab.items()}

    def encode(self, input_sequence):
        return [self.vocab[token] for token in input_sequence]

    def decode(self, input_ids):
        return [self.itos[idx] for idx in input_ids]

    def __len__(self):
        return len(self.vocab)


class DateDataset(Dataset):
    def __init__(self, src_dates, tgt_dates, src_vocab, tgt_vocab):
        self.samples = []
        for src_date, tgt_date in zip(src_dates, tgt_dates):
            src_ids = src_vocab.encode(src_date)
            tgt_ids = [SOS_IDX] + tgt_vocab.encode(tgt_date) + [EOS_IDX]
            self.samples.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


def build_loaders(src_dates, tgt_dates, batch_size=32, train_size=3000):
    src_vocab = Vocab(src_dates, special_tokens)
    tgt_vocab = Vocab(tgt_dates, special_tokens)
    train_set = DateDataset(src_dates[:train_size], tgt_dates[:train_size],
                            src_vocab, tgt_vocab)
    valid_set = DateDataset(src_dates[train_size:], tgt_dates[train_size:],
                            src_vocab, tgt_vocab)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn)
    return src_vocab, tgt_vocab, train_loader, valid_loader


src_vocab, tgt_vocab, train_loader, valid_loader = build_loaders(src_dates, tgt_dates)
print(f'입력 어휘 사전 {len(src_vocab)}, 출력 어휘 사전 {len(tgt_vocab)}')

입력 어휘 사전 43, 출력 어휘 사전 14


In [4]:
# Seq2Seq 모델 ([코드 9-3] ~ [코드 9-8], [코드 9-16]의 패킹 적용)
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers,
                 use_packing=True):
        super().__init__()
        self.use_packing = use_packing
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)

    def forward(self, src, src_lengths):
        embedded = self.encoder_embedding(src)
        if self.use_packing:
            packed = pack_padded_sequence(embedded, src_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
            _, (hidden, cell) = self.encoder_lstm(packed)
        else:
            _, (hidden, cell) = self.encoder_lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.decoder_lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward_step(self, token, hidden, cell, context):
        embedded = self.decoder_embedding(token)
        rnn_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.decoder_lstm(rnn_input, (hidden, cell))
        logits = self.fc(output.squeeze(1))
        return logits, hidden, cell

    def forward(self, tgt, hidden, cell, context, forcing_ratio=0.5):
        _, target_length = tgt.shape
        all_logits = []
        token = tgt[:, 0:1]
        for i in range(target_length):
            logits, hidden, cell = self.forward_step(token, hidden, cell, context)
            all_logits.append(logits.unsqueeze(1))
            if i + 1 < target_length:
                if random.random() < forcing_ratio:
                    token = tgt[:, i + 1:i + 2]
                else:
                    token = logits.argmax(dim=-1, keepdim=True)
        return torch.cat(all_logits, dim=1)


class DateConverter(nn.Module):
    def __init__(self, encoder, decoder, forcing_ratio=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.forcing_ratio = forcing_ratio

    def forward(self, src, tgt, src_lengths):
        hidden, cell = self.encoder(src, src_lengths)
        context = hidden[-1].unsqueeze(1)
        tgt_input = tgt[:, :-1]
        return self.decoder(tgt_input, hidden, cell, context,
                            forcing_ratio=self.forcing_ratio)

In [5]:
# 학습, 검증, 예측 함수 ([코드 9-10], [코드 9-11])
import copy

import torch.optim as optim

EMBED_DIM, HIDDEN_DIM, NUM_LAYERS = 32, 32, 1
EPOCHS, PATIENCE, LR = 120, 5, 1e-3
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, token_count = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        optimizer.zero_grad()
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
    return total_loss / token_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """토큰 단위 손실과 샘플 단위 정확도(모든 토큰이 일치할 때만 정답)를 반환

    본문 예제와 마찬가지로 검증에는 교사 강제를 적용하지 않는다.
    """
    model.eval()
    saved_ratio = model.forcing_ratio
    model.forcing_ratio = 0.0
    total_loss, token_count = 0.0, 0
    correct, total = 0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
        pred = logits.argmax(dim=-1)
        # <pad> 위치를 제외하고 모든 토큰이 일치해야 정답
        valid = labels != PAD_IDX
        match = ((pred == labels) | ~valid).all(dim=1)
        correct += match.sum().item()
        total += labels.size(0)
    model.forcing_ratio = saved_ratio
    return total_loss / token_count, correct / total * 100


def train_model(model, train_loader, valid_loader, name,
                epochs=EPOCHS, patience=PATIENCE, lr=LR, verbose_rows=8,
                optimizer_groups=None):
    """optimizer_groups를 주면 7장 [코드 7-9]처럼 계층별로 다른 학습률을 적용한다"""
    model.to(device)
    optimizer = (optim.Adam(optimizer_groups) if optimizer_groups
                 else optim.Adam(model.parameters(), lr=lr))
    log = common.EpochLogger(epochs, target_rows=verbose_rows)
    best_loss, best_epoch, best_state, counter = float('inf'), -1, None, 0
    print(f'{name} 학습')
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_loss:
            best_loss, best_epoch = valid_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    log.summary(stopped='조기 종료' if counter >= patience else None)
    if best_state is not None:
        model.load_state_dict(best_state)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
    print(f'{name}: 최적 에포크 {best_epoch}, 검증 손실 {valid_loss:.4f}, '
          f'검증 정확도 {valid_acc:.2f}%')
    return {'name': name, 'best_epoch': best_epoch,
            'valid_loss': valid_loss, 'valid_acc': valid_acc}


@torch.no_grad()
def predict(model, src_text, src_vocab, tgt_vocab, max_length=12):
    model.eval()
    src_ids = src_vocab.encode(src_text)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src_ids)])
    hidden, cell = model.encoder(src, src_lengths)
    context = hidden[-1].unsqueeze(1)
    token = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
    result = []
    for _ in range(max_length):
        logits, hidden, cell = model.decoder.forward_step(token, hidden, cell, context)
        pred = logits.argmax(dim=-1, keepdim=True)
        if pred.item() == EOS_IDX:
            break
        result.append(pred.item())
        token = pred
    return ''.join(tgt_vocab.decode(result))


def build_model(src_vocab, tgt_vocab, forcing_ratio=0.5, use_packing=True):
    common.set_seed(SEED, deterministic=True)
    encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS,
                      use_packing=use_packing)
    decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
    return DateConverter(encoder, decoder, forcing_ratio=forcing_ratio)

## 공통 준비 — 어텐션 모델 ([코드 9-18] ~ [코드 9-20])

In [6]:
# 어텐션 계층과 어텐션을 적용한 인코더, 디코더, 모델
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn_energy = nn.Linear(hidden_dim * 2, hidden_dim)
        self.score_projection = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_output, pad_mask=None):
        src_length = encoder_output.shape[1]
        hidden_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_length, 1)
        combined = torch.cat([hidden_expanded, encoder_output], dim=-1)
        energy = torch.tanh(self.attn_energy(combined))
        scores = self.score_projection(energy).squeeze(-1)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask == 0, -float('inf'))
        return torch.softmax(scores, dim=1)


class AttnEncoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)

    def forward(self, src, src_lengths):
        embedded = self.encoder_embedding(src)
        packed = pack_padded_sequence(embedded, src_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.encoder_lstm(packed)
        encoder_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        return encoder_output, hidden, cell


class AttnDecoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.attention = Attention(hidden_dim)
        self.decoder_lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward_step(self, token, hidden, cell, encoder_output, pad_mask):
        embedded = self.decoder_embedding(token)
        weights = self.attention(hidden[-1], encoder_output, pad_mask)
        context = torch.bmm(weights.unsqueeze(1), encoder_output)
        lstm_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.decoder_lstm(lstm_input, (hidden, cell))
        return self.fc(output.squeeze(1)), hidden, cell, weights

    def forward(self, tgt, hidden, cell, encoder_output, pad_mask=None,
                forcing_ratio=0.5):
        _, target_length = tgt.shape
        all_logits = []
        token = tgt[:, 0:1]
        for i in range(target_length):
            logits, hidden, cell, _ = self.forward_step(
                token, hidden, cell, encoder_output, pad_mask)
            all_logits.append(logits.unsqueeze(1))
            if i + 1 < target_length:
                if random.random() < forcing_ratio:
                    token = tgt[:, i + 1:i + 2]
                else:
                    token = logits.argmax(dim=-1, keepdim=True)
        return torch.cat(all_logits, dim=1)


class DateConverterAttn(nn.Module):
    def __init__(self, encoder, decoder, forcing_ratio=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.forcing_ratio = forcing_ratio

    def forward(self, src, tgt, src_lengths):
        encoder_output, hidden, cell = self.encoder(src, src_lengths)
        pad_mask = (src != PAD_IDX)
        tgt_input = tgt[:, :-1]
        return self.decoder(tgt_input, hidden, cell, encoder_output, pad_mask,
                            forcing_ratio=self.forcing_ratio)


def build_attn_model(src_vocab, tgt_vocab, forcing_ratio=0.5):
    common.set_seed(SEED, deterministic=True)
    encoder = AttnEncoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
    decoder = AttnDecoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
    return DateConverterAttn(encoder, decoder, forcing_ratio=forcing_ratio)


@torch.no_grad()
def predict_attn(model, src_text, src_vocab, tgt_vocab, max_length=12,
                 return_weights=False):
    model.eval()
    src_ids = src_vocab.encode(src_text)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src_ids)])
    encoder_output, hidden, cell = model.encoder(src, src_lengths)
    pad_mask = (src != PAD_IDX)
    token = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
    result, all_weights = [], []
    for _ in range(max_length):
        logits, hidden, cell, weights = model.decoder.forward_step(
            token, hidden, cell, encoder_output, pad_mask)
        pred = logits.argmax(dim=-1, keepdim=True)
        if pred.item() == EOS_IDX:
            break
        result.append(pred.item())
        all_weights.append(weights.squeeze(0).cpu())
        token = pred
    text = ''.join(tgt_vocab.decode(result))
    return (text, all_weights) if return_weights else text


print('어텐션 모델 정의 완료')

어텐션 모델 정의 완료


---

## 연습 문제 9-9

> 1에서 1,000 사이의 정수 10개를 무작위로 뽑아 쉼표로 구분한 문자열을 입력하면, 오름차순으로 정렬한 문자열을 생성하는 **어텐션을 적용한** Seq2Seq 모델을 만들어 보자. 이 문제는 9-2절 [연습 문제 9-7]에 어텐션을 적용해 보는 문제다.

In [7]:
# [연습 문제 9-7]과 같은 데이터
NUM_COUNT = 10
SORT_EPOCHS = 60


def generate_sort_pairs(sample_size, num_count=NUM_COUNT, low=1, high=1000):
    src_list, tgt_list = [], []
    for _ in range(sample_size):
        numbers = [random.randint(low, high) for _ in range(num_count)]
        src_list.append(', '.join(str(n) for n in numbers))
        tgt_list.append(', '.join(str(n) for n in sorted(numbers)))
    return src_list, tgt_list


common.set_seed(SEED, deterministic=True)
sort_src, sort_tgt = generate_sort_pairs(4000)
s_src_vocab, s_tgt_vocab, s_train, s_valid = build_loaders(sort_src, sort_tgt)
print(f'입력: {sort_src[0]}')
print(f'정답: {sort_tgt[0]}')

results = {}
model_sort_base = build_model(s_src_vocab, s_tgt_vocab)
# 9-7과 같은 이유로 조기 종료를 끄고 고정 예산을 모두 사용한다.
results['sort_base'] = train_model(model_sort_base, s_train, s_valid,
                                   '정렬(어텐션 없음)', epochs=SORT_EPOCHS,
                                   patience=SORT_EPOCHS)

입력: 655, 115, 26, 760, 282, 251, 229, 143, 755, 105
정답: 26, 105, 115, 143, 229, 251, 282, 655, 755, 760


정렬(어텐션 없음) 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       2.4052       2.4487        0.00%     0:06


  8/60       1.3625       3.1703        0.00%     0:41


 16/60       1.2907       3.3669        0.00%     1:21


...

 40/60       1.1029       1.3688        0.00%     3:21


 48/60       1.0685       1.2541        0.00%     4:01


 56/60       1.0353       1.2412        0.00%     4:41


 60/60       1.0284       1.1675        0.00%     5:00
------------------------------------------------------
최적 57 에포크 · 검증 손실 1.1618 · 전체 학습 시간 5:00


정렬(어텐션 없음): 최적 에포크 57, 검증 손실 1.1618, 검증 정확도 0.00%


In [8]:
model_sort_attn = build_attn_model(s_src_vocab, s_tgt_vocab)
results['sort_attn'] = train_model(model_sort_attn, s_train, s_valid,
                                   '정렬(어텐션 적용)', epochs=SORT_EPOCHS,
                                   patience=SORT_EPOCHS)

정렬(어텐션 적용) 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       2.3659       2.5379        0.00%     0:09


  8/60       1.3835       3.4412        0.00%     1:15


 16/60       1.2815       3.0396        0.00%     2:30


 24/60       1.1659       1.7725        0.00%     3:46


...

 40/60       0.6487       0.9434        0.00%     6:15


 48/60       0.4437       0.8704        0.50%     7:32


 56/60       0.3553       0.8001        1.80%     8:50


 60/60       0.3181       0.8143        2.00%     9:28
------------------------------------------------------
최적 59 에포크 · 검증 손실 0.6461 · 전체 학습 시간 9:28


정렬(어텐션 적용): 최적 에포크 59, 검증 손실 0.6461, 검증 정확도 2.40%


In [9]:
# 세 가지 지표로 비교한다
@torch.no_grad()
def sort_report(predict_fn, model, src_list, tgt_list, src_vocab, tgt_vocab, n=200):
    exact, hit, total, valid = 0, 0, 0, 0
    for text, answer in zip(src_list[:n], tgt_list[:n]):
        out = predict_fn(model, text, src_vocab, tgt_vocab, max_length=60)
        exact += (out == answer)
        for a, b in zip(out, answer):
            hit += (a == b)
        total += len(answer)
        try:
            nums = [int(x) for x in out.split(',')]
            valid += (nums == sorted(nums))
        except ValueError:
            pass
    return exact / n * 100, hit / total * 100, valid / n * 100


base_m = sort_report(predict, model_sort_base, sort_src[3000:], sort_tgt[3000:],
                     s_src_vocab, s_tgt_vocab)
attn_m = sort_report(predict_attn, model_sort_attn, sort_src[3000:], sort_tgt[3000:],
                     s_src_vocab, s_tgt_vocab)

print(f"{'모델':<18}{'검증 정확도':>12}{'완전 일치':>12}{'글자 일치':>12}{'정렬된 출력':>14}")
print('-' * 70)
for label, r, m in (('어텐션 없음', results['sort_base'], base_m),
                    ('어텐션 적용', results['sort_attn'], attn_m)):
    print(f'{label:<18}{r["valid_acc"]:>11.2f}%{m[0]:>11.1f}%{m[1]:>11.1f}%{m[2]:>13.1f}%')

print()
print('생성 예시(어텐션 적용)')
for text, answer in list(zip(sort_src[3000:], sort_tgt[3000:]))[:2]:
    out = predict_attn(model_sort_attn, text, s_src_vocab, s_tgt_vocab, max_length=60)
    print(f'  입력: {text}')
    print(f'  정답: {answer}')
    print(f'  생성: {out}')

모델                      검증 정확도       완전 일치       글자 일치        정렬된 출력
----------------------------------------------------------------------
어텐션 없음                   0.00%        0.0%       58.5%         74.0%
어텐션 적용                   2.40%        3.0%       85.4%         55.5%

생성 예시(어텐션 적용)
  입력: 287, 922, 555, 181, 798, 765, 351, 873, 384, 275
  정답: 181, 275, 287, 351, 384, 555, 765, 798, 873, 922
  생성: 181, 287, 287, 351, 384, 555, 765, 798, 873, 922
  입력: 258, 187, 957, 537, 403, 472, 608, 404, 201, 144
  정답: 144, 187, 201, 258, 403, 404, 472, 537, 608, 957
  생성: 144, 187, 201, 258, 403, 403, 472, 537, 608


### 풀이 해설 — 연습 문제 9-9

**구현은 [연습 문제 9-7]에서 모델 클래스만 바꾸면 끝난다.** 데이터, 어휘 사전, 학습 함수가 모두
그대로다. 9-3절 본문이 "어텐션 메커니즘은 클래스 내부에만 적용되어 기본 Seq2Seq 모델을 학습하는
학습 함수를 그대로 사용할 수 있다"고 한 그대로다. [연습 문제 9-7]과 같은 이유로 조기 종료는
끄고 60 에포크를 모두 사용한다.

**실행 결과를 지표별로 나눠 보면 두 모델이 서로 다른 것을 배웠다는 것이 드러난다.**

| 모델 | 완전 일치 | 글자 일치 | 출력이 정렬된 비율 |
|---|---|---|---|
| 어텐션 없음 | 0.0% | 58.5% | **74.0%** |
| 어텐션 적용 | 3.0% | **85.4%** | 55.5% |

**글자 일치율이 58.5%에서 85.4%로 크게 올랐다.** 생성 예시를 보면 무엇이 달라졌는지 분명하다.

```
입력: 287, 922, 555, 181, 798, 765, 351, 873, 384, 275
정답: 181, 275, 287, 351, 384, 555, 765, 798, 873, 922

어텐션 없음: 177, 227, 277, 327, 377, 527, 727, 787, 827, 977
어텐션 적용: 181, 287, 287, 351, 384, 555, 765, 798, 873, 922
```

어텐션이 없으면 **입력에 없던 숫자를 지어낸다.** 오름차순이라는 형태만 갖춘 채 내용은 전부
엉터리다. 어텐션을 붙이면 **열 개 중 아홉 개를 정확히 집어 온다.** 콘텍스트 벡터 하나에 갇혀
있던 입력 정보에 디코더가 직접 접근하게 된 결과다.

**'출력이 정렬된 비율'이 74.0%에서 55.5%로 오히려 떨어진 것이 흥미롭다.** 언뜻 나빠진 것처럼
보이지만, 두 모델이 실패하는 방식이 다르기 때문이다.

- **어텐션 없음**: 숫자를 모르니 **안전하게 오름차순 형태만 지킨다.** 입력과 무관한 답이므로
  정렬 여부는 쉽게 지킬 수 있다.
- **어텐션 적용**: 실제 숫자를 가져오려다 **가끔 잘못된 위치를 집는다.** 위 예시에서도 `275`를
  집어야 할 자리에 `287`을 집어 같은 숫자가 두 번 나왔다.

**즉 '정렬된 비율'이 떨어진 것은 모델이 더 어려운 일을 시도했기 때문이다.** 아무 숫자나 순서대로
늘어놓는 것과 실제 숫자를 찾아오는 것 중 어느 쪽이 정렬 문제를 푼 것인지는 분명하다. 지표 하나만
보면 결론이 뒤집힐 수 있다는 것도 이 문제의 수확이다.

**완전 일치는 여전히 3.0%로 낮다.** 열 개를 모두 맞혀야 하므로 하나만 틀려도 오답이다. 개당
정확도가 80%라면 열 개를 모두 맞힐 확률은 `0.8^10 = 10.7%`에 불과하다. 어텐션이 정보 병목을
크게 완화했지만, 순환 신경망으로 40~50자 길이의 전역 비교를 푸는 데는 여전히 한계가 있다.
그 다음 답이 10장의 트랜스포머다.


---

## 연습 문제 9-10

> 영어 형식의 날짜 문자열을 한국어 날짜 문자열로 변환하는 Seq2Seq 모델을 어텐션을 적용해 만들어 보자. 입력은 9장 예제에서 사용한 여덟 가지 영어 형식을 그대로 사용하고, 정답은 `이천이십육년 이월 일일`과 같은 한국어 날짜 표현이다.

In [10]:
# 한국어 날짜 표현 생성
DIGITS = '영일이삼사오육칠팔구'


def year_to_korean(year):
    """1900 -> '천구백', 2026 -> '이천이십육' (자릿수 읽기)"""
    units = [(1000, '천'), (100, '백'), (10, '십')]
    text = ''
    rest = year
    for value, name in units:
        digit, rest = divmod(rest, value)
        if digit:
            # 1은 '일천'이 아니라 '천'으로 읽는다
            text += ('' if digit == 1 else DIGITS[digit]) + name
    if rest:
        text += DIGITS[rest]
    return text or '영'


def small_to_korean(number):
    """1 -> '일', 12 -> '십이', 25 -> '이십오'"""
    tens, ones = divmod(number, 10)
    text = ''
    if tens:
        text += ('' if tens == 1 else DIGITS[tens]) + '십'
    if ones:
        text += DIGITS[ones]
    return text or '영'


# 월 이름에는 불규칙 읽기가 있다: 6월은 '유월', 10월은 '시월'
IRREGULAR_MONTHS = {6: '유', 10: '시'}


def month_to_korean(month):
    return IRREGULAR_MONTHS.get(month, small_to_korean(month))


def date_to_korean(date):
    return (f'{year_to_korean(date.year)}년 '
            f'{month_to_korean(date.month)}월 '
            f'{small_to_korean(date.day)}일')


def generate_korean_datepairs(date_list, src_formats=src_formats):
    src_list, tgt_list = [], []
    for i, date in enumerate(date_list):
        src_list.append(date.strftime(src_formats[i % len(src_formats)]))
        tgt_list.append(date_to_korean(date))
    return src_list, tgt_list


common.set_seed(SEED, deterministic=True)
ko_date_list = choice_random_dates(4000)
ko_src, ko_tgt = generate_korean_datepairs(ko_date_list)
print('변환 예시')
for i in range(4):
    print(f'  {ko_src[i]:<20} -> {ko_tgt[i]}')
print()
print(f'출력 길이: 최소 {min(len(t) for t in ko_tgt)}, 최대 {max(len(t) for t in ko_tgt)}')

변환 예시
  25 September 2014    -> 이천십사년 구월 이십오일
  24 Dec 1919          -> 천구백십구년 십이월 이십사일
  June 28, 1904        -> 천구백사년 유월 이십팔일
  Jan 21, 2033         -> 이천삼십삼년 일월 이십일일

출력 길이: 최소 9, 최대 16


In [11]:
ko_src_vocab, ko_tgt_vocab, ko_train, ko_valid = build_loaders(ko_src, ko_tgt)
print(f'입력 어휘 사전 {len(ko_src_vocab)}, 출력 어휘 사전 {len(ko_tgt_vocab)}')
print(f'출력 토큰: {sorted(ko_tgt_vocab.vocab)}')

model_ko = build_attn_model(ko_src_vocab, ko_tgt_vocab)
results['korean'] = train_model(model_ko, ko_train, ko_valid, '한국어 날짜 변환(어텐션)')

입력 어휘 사전 43, 출력 어휘 사전 20
출력 토큰: [' ', '<eos>', '<pad>', '<sos>', '구', '년', '백', '사', '삼', '시', '십', '오', '월', '유', '육', '이', '일', '천', '칠', '팔']
한국어 날짜 변환(어텐션) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.4550       1.9914        0.00%     0:04


 15/120       0.3172       0.3667       26.30%     0:53


 30/120       0.0145       0.0129       99.40%     1:46


 45/120       0.0027       0.0031       99.90%     2:39


...

 75/120       0.0005       0.0006      100.00%     4:28


 90/120       0.0002       0.0003      100.00%     5:21


105/120       0.0001       0.0002      100.00%     6:15


120/120       0.0001       0.0001      100.00%     7:09
-------------------------------------------------------
최적 120 에포크 · 검증 손실 0.0001 · 전체 학습 시간 7:09


한국어 날짜 변환(어텐션): 최적 에포크 120, 검증 손실 0.0001, 검증 정확도 100.00%


In [12]:
print('생성 결과')
for text, answer in list(zip(ko_src[3000:], ko_tgt[3000:]))[:8]:
    out = predict_attn(model_ko, text, ko_src_vocab, ko_tgt_vocab, max_length=24)
    mark = '정답' if out == answer else '오답'
    print(f'  {text:<20} -> {out:<22} ({mark}, 정답 {answer})')

생성 결과
  27 July 2000         -> 이천년 칠월 이십칠일            (정답, 정답 이천년 칠월 이십칠일)
  20 Sep 2042          -> 이천사십이년 구월 이십일          (정답, 정답 이천사십이년 구월 이십일)
  June 27, 2029        -> 이천이십구년 유월 이십칠일         (정답, 정답 이천이십구년 유월 이십칠일)
  Jan 25, 1949         -> 천구백사십구년 일월 이십오일        (정답, 정답 천구백사십구년 일월 이십오일)
  08/22/1951           -> 천구백오십일년 팔월 이십이일        (정답, 정답 천구백오십일년 팔월 이십이일)
  1921/11/14           -> 천구백이십일년 십일월 십사일        (정답, 정답 천구백이십일년 십일월 십사일)
  25-11-2002           -> 이천이년 십일월 이십오일          (정답, 정답 이천이년 십일월 이십오일)
  2021-05-29           -> 이천이십일년 오월 이십구일         (정답, 정답 이천이십일년 오월 이십구일)


### 풀이 해설 — 연습 문제 9-10

**데이터를 만드는 일이 이 문제의 절반이다.** 한국어 숫자 읽기에는 규칙이 있다.

- **`1`을 읽지 않는 자리가 있다.** 2026년은 '이천이십육'이지 '이천이십일육'이 아니다. 이 풀이는 십의 자리와 천의 자리에서 `1`을 생략하도록 처리했다(`1900` → '천구백').
- **월 이름에는 불규칙 읽기가 있다.** 6월은 '육월'이 아니라 **'유월'**, 10월은 '십월'이 아니라 **'시월'**이다. 일(日)에는 이런 불규칙이 없어 6일은 '육일', 10일은 '십일'이다. 딕셔너리 하나로 예외를 처리했다.
- **연도와 월·일의 읽기가 다르다.** 연도는 네 자리를 자릿수로 읽고(`2026` → '이천이십육'), 월과 일은 두 자리만 읽는다.

**모델 쪽에서 달라지는 것은 어휘 사전뿐이다.** 출력 어휘 사전이 숫자·하이픈 11개에서 한글 숫자와 '년/월/일', 공백을 포함한 스무 개 남짓으로 바뀐다. 모델 코드는 한 줄도 고칠 필요가 없다. **Seq2Seq가 입력과 출력을 분리해 다루기 때문**이며, 9-1절이 "어휘 사전과 임베딩을 입력용과 출력용으로 구분해 만드는 것이 일반적"이라고 한 설계가 여기서 값을 한다.

**날짜 변환보다 어려운 점이 두 가지 있다.**

1. **출력이 길다.** `2026-2-1`은 8자인데 '이천이십육년 이월 일일'은 12자다. 생성 단계가 늘어나면 오차가 누적될 구간도 길어진다.
2. **출력 길이가 들쭉날쭉하다.** '천구백년 일월 일일'(10자)부터 '이천사십팔년 십이월 이십팔일'(15자)까지 편차가 크다. 날짜 변환의 출력(`YYYY-M-D`, 8~10자)보다 변동이 심하다.

그래서 어텐션의 값어치가 날짜 변환보다 크다. 실행 결과는 위 표로 확인한다.

---

## 연습 문제 9-11 [도전 문제]

> 이번에는 전이 학습으로 [연습 문제 9-10]을 풀어 보자. 먼저 9-3절의 예제 모델을 노이즈 없는 데이터로 학습해 사전 학습 모델을 만든다. 그런 다음 이 모델을 출발점으로 [연습 문제 9-10]에서 제시한 언어 간 날짜 문자열 변환을 전이 학습한다. 전이 학습에는 이전 문제에서 사용한 데이터의 10%만 사용해 어느 정도 성능에 이르는지 확인해 보자.

In [13]:
# 1단계 - 사전 학습: 노이즈 없는 영어 -> 숫자 날짜 변환
pre_src_vocab, pre_tgt_vocab, pre_train, pre_valid = build_loaders(src_dates, tgt_dates)
model_pre = build_attn_model(pre_src_vocab, pre_tgt_vocab)
results['pretrain'] = train_model(model_pre, pre_train, pre_valid, '사전 학습(영어 -> 숫자)')

사전 학습(영어 -> 숫자) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.1801       1.8086        0.00%     0:03


 15/120       0.2897       0.3005       58.90%     0:39


 30/120       0.0118       0.0266       97.60%     1:17


 45/120       0.0024       0.0074       99.60%     1:56


 60/120       0.0010       0.0028       99.80%     2:35


 75/120       0.0004       0.0027       99.70%     3:13


-------------------------------------------------------
최적 72 에포크 · 검증 손실 0.0025 · 전체 학습 시간 3:18 · (조기 종료)


사전 학습(영어 -> 숫자): 최적 에포크 72, 검증 손실 0.0025, 검증 정확도 99.70%


In [14]:
# 2단계 - 전이 학습 데이터: [연습 문제 9-10] 데이터의 10%
TRANSFER_TRAIN = 300        # 3,000개의 10%
TRANSFER_VALID = 1000

# 입력 어휘 사전은 사전 학습 모델의 것을 그대로 쓴다(같은 영어 날짜 형식)
print(f'사전 학습 입력 어휘 사전 {len(pre_src_vocab)}, '
      f'한국어 문제 입력 어휘 사전 {len(ko_src_vocab)}')
print(f'두 입력 어휘 사전이 같은가: {pre_src_vocab.vocab == ko_src_vocab.vocab}')

small_train = DateDataset(ko_src[:TRANSFER_TRAIN], ko_tgt[:TRANSFER_TRAIN],
                          pre_src_vocab, ko_tgt_vocab)
small_valid = DateDataset(ko_src[3000:3000 + TRANSFER_VALID],
                          ko_tgt[3000:3000 + TRANSFER_VALID],
                          pre_src_vocab, ko_tgt_vocab)
small_train_loader = DataLoader(small_train, batch_size=32, shuffle=True,
                                collate_fn=collate_fn)
small_valid_loader = DataLoader(small_valid, batch_size=32, shuffle=False,
                                collate_fn=collate_fn)
print(f'전이 학습 훈련 {len(small_train)}개, 검증 {len(small_valid)}개')

사전 학습 입력 어휘 사전 43, 한국어 문제 입력 어휘 사전 43
두 입력 어휘 사전이 같은가: True
전이 학습 훈련 300개, 검증 1000개


In [15]:
# 3단계 - 전이 학습 모델: 인코더는 물려받고, 디코더는 출력 어휘 사전이 달라 새로 만든다
import copy as copy_module

common.set_seed(SEED, deterministic=True)
transfer_encoder = copy_module.deepcopy(model_pre.encoder)     # 사전 학습된 인코더
transfer_decoder = AttnDecoder(len(ko_tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
model_transfer = DateConverterAttn(transfer_encoder, transfer_decoder)

print(f'인코더 파라미터 {common.count_params(transfer_encoder):,}개 (사전 학습 값 물려받음)')
print(f'디코더 파라미터 {common.count_params(transfer_decoder):,}개 (새로 초기화)')

# 사전 학습된 인코더에는 낮은 학습률, 새 디코더에는 통상 학습률(7장 [코드 7-9] 방식)
LR_FT = 1e-4
optimizer_groups = [
    {'params': model_transfer.encoder.parameters(), 'lr': LR_FT},
    {'params': model_transfer.decoder.parameters(), 'lr': LR},
]
results['transfer'] = train_model(
    model_transfer, small_train_loader, small_valid_loader,
    '전이 학습(데이터 10%)', optimizer_groups=optimizer_groups)

인코더 파라미터 9,824개 (사전 학습 값 물려받음)
디코더 파라미터 15,956개 (새로 초기화)
전이 학습(데이터 10%) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.9166       2.8327        0.00%     0:01


 15/120       1.6359       1.7533        0.00%     0:11


 30/120       1.0869       1.2605        0.00%     0:22


 45/120       0.7893       0.9711        0.00%     0:32


...

 75/120       0.4114       0.6293       11.60%     0:53


 90/120       0.2431       0.3691       49.30%     1:04


105/120       0.1364       0.2515       82.40%     1:15


120/120       0.0744       0.1892       89.60%     1:25
-------------------------------------------------------
최적 120 에포크 · 검증 손실 0.1892 · 전체 학습 시간 1:25


전이 학습(데이터 10%): 최적 에포크 120, 검증 손실 0.1892, 검증 정확도 89.60%


In [16]:
# 4단계 - 비교 대상: 같은 10% 데이터로 처음부터 학습한 모델
model_scratch = build_attn_model(pre_src_vocab, ko_tgt_vocab)
results['scratch'] = train_model(model_scratch, small_train_loader, small_valid_loader,
                                 '처음부터 학습(데이터 10%)')

처음부터 학습(데이터 10%) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.9623       2.8971        0.00%     0:01


 15/120       1.6165       1.7652        0.00%     0:11


 30/120       1.1678       1.4546        0.00%     0:21


 45/120       0.9125       1.2759        0.00%     0:32


...

 75/120       0.7000       1.0439        0.10%     0:54


 90/120       0.6004       1.0144        0.70%     1:04


105/120       0.4945       0.9118        1.90%     1:16


-------------------------------------------------------
최적 103 에포크 · 검증 손실 0.9041 · 전체 학습 시간 1:19 · (조기 종료)


처음부터 학습(데이터 10%): 최적 에포크 103, 검증 손실 0.9041, 검증 정확도 1.60%


In [17]:
print(f"{'모델':<26}{'훈련 샘플':>10}{'최적 에포크':>10}{'검증 정확도':>14}")
print('-' * 62)
rows = [('전체 데이터(문제 9-10)', 3000, results['korean']),
        ('전이 학습(10%)', TRANSFER_TRAIN, results['transfer']),
        ('처음부터 학습(10%)', TRANSFER_TRAIN, results['scratch'])]
for label, n, r in rows:
    print(f'{label:<26}{n:>10}{r["best_epoch"]:>10}{r["valid_acc"]:>13.2f}%')

print()
print('전이 학습 모델의 생성 결과')
for text, answer in list(zip(ko_src[3000:], ko_tgt[3000:]))[:5]:
    out = predict_attn(model_transfer, text, pre_src_vocab, ko_tgt_vocab, max_length=24)
    mark = '정답' if out == answer else '오답'
    print(f'  {text:<20} -> {out:<22} ({mark})')

모델                             훈련 샘플    최적 에포크        검증 정확도
--------------------------------------------------------------
전체 데이터(문제 9-10)                 3000       120       100.00%
전이 학습(10%)                       300       120        89.60%
처음부터 학습(10%)                     300       103         1.60%

전이 학습 모델의 생성 결과
  27 July 2000         -> 이천년 칠월 이십칠일            (정답)
  20 Sep 2042          -> 이천사십이년 구월 이십일          (정답)
  June 27, 2029        -> 이천이십구년 유월 이십칠일         (정답)
  Jan 25, 1949         -> 천구백사십구년 일월 이십오일        (정답)
  08/22/1951           -> 천구백오십일년 팔월 이십이일        (정답)


### 풀이 해설 — 연습 문제 9-11

**이 문제의 핵심은 "무엇을 물려받을 수 있는가"를 가려내는 것이다.**

| 구성 요소 | 물려받을 수 있나 | 이유 |
|---|---|---|
| **인코더**(임베딩 + LSTM) | **그렇다** | 입력이 동일한 영어 날짜 문자열이다. 입력 어휘 사전도 같다 |
| 디코더 임베딩 | 아니다 | 출력 어휘 사전이 숫자에서 한글로 완전히 바뀐다 |
| 디코더 LSTM | 원칙적으로 가능 | 그러나 임베딩과 분류기가 바뀌므로 함께 새로 학습하는 편이 낫다 |
| 분류기(`fc`) | 아니다 | 출력 크기가 어휘 사전 크기와 같아 형태부터 다르다 |

**그래서 이 전이 학습은 '인코더만 물려받는' 형태가 된다.** 7장의 전이 학습(오토인코더 인코더를 분류기에 재활용)과 구조가 같다. 8-4절에서 ResNet-50의 분류기만 갈아 끼운 것과도 같은 발상이다.

**인코더를 물려받는 것이 왜 값어치가 있는가.** 인코더가 이미 배운 것은 **'여덟 가지 영어 날짜 형식에서 연·월·일을 찾아내는 방법'**이다. 출력 언어가 한국어로 바뀌어도 이 능력은 그대로 쓸모가 있다. 새로 배워야 하는 것은 **'찾아낸 연·월·일을 한국어로 읽는 방법'**뿐이고, 그것은 디코더의 몫이다.

**학습률을 둘로 나눈 것**은 7장 [코드 7-9]의 방식이다. 사전 학습된 인코더는 `1e-4`로 살짝만 움직이고, 새로 만든 디코더는 `1e-3`으로 빠르게 배운다. 같은 학습률을 쓰면 첫 몇 배치에서 인코더가 크게 흔들려 물려받은 능력을 잃는다.

**비교 대상을 셋 둔 이유**가 있다. '전이 학습(10%)'만 보면 잘한 것인지 알 수 없다.

- **전체 데이터(100%)** — 도달 가능한 상한선
- **전이 학습(10%)** — 이 문제가 만든 것
- **처음부터 학습(10%)** — 전이 학습이 없었다면 얻었을 성능

전이 학습이 '처음부터 학습(10%)'보다 얼마나 앞서는지가 **인코더를 물려받아 얻은 이득**이고, '전체 데이터'와 얼마나 벌어지는지가 **데이터 90%를 포기한 대가**다. 실행 결과는 위 표로 확인한다.